# GraphRouter - Training

This notebook demonstrates how to train the **GraphRouter** (Graph Neural Network Router).

## Overview

GraphRouter uses a Graph Neural Network (GNN) to model the relationships between queries and LLMs.
It constructs a heterogeneous graph where queries and LLMs are nodes, and performance scores are edge weights.

**Key Features**:
- Graph-based representation of query-LLM interactions
- Message passing for learning representations
- Can capture complex relational patterns

## 1. Environment Setup

In [ ]:
# Install required packages (for Colab)
!pip install llmrouter-lib torch torch-geometric
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
import torch
from llmrouter.models.graphrouter import GraphRouter, GraphTrainer
from llmrouter.utils import setup_environment

setup_environment()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Configuration

GraphRouter uses the following configuration parameters:

| Parameter | Description | Default |
|-----------|-------------|--------|
| `hidden_dim` | GNN hidden layer dimension | 64 |
| `learning_rate` | Learning rate | 0.001 |
| `weight_decay` | L2 regularization | 0.0001 |
| `train_epoch` | Training epochs | 100 |
| `batch_size` | Batch size | 4 |
| `train_mask_rate` | Edge masking rate | 0.3 |
| `val_split_ratio` | Validation split | 0.2 |

In [ ]:
import yaml

CONFIG_PATH = "configs/model_config_train/graphrouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

## 3. Initialize Router

In [ ]:
router = GraphRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Number of training samples: {len(router.routing_data_train)}")
print(f"Number of LLM candidates: {len(router.llm_data)}")
print(f"LLM candidates: {list(router.llm_data.keys())}")

## 4. Graph Structure Visualization

In [ ]:
# Understand the graph structure
print("Graph Structure Information:")
print("=" * 50)
print(f"\nNode types:")
print(f"  - Query nodes: Based on training queries")
print(f"  - LLM nodes: {len(router.llm_data)} models")
print(f"\nEdge types:")
print(f"  - Query -> LLM edges (performance scores)")
print(f"\nThe GNN learns to predict missing edges for new queries.")

## 5. Training

In [ ]:
trainer = GraphTrainer(router=router, device=device)

print("Trainer initialized!")
print(f"Device: {device}")
print(f"Save path: {trainer.save_model_path}")

In [ ]:
print("Starting training...")
print("=" * 50)

best_result = trainer.train()

print("=" * 50)
print("Training completed!")
if best_result:
    print(f"Best validation result: {best_result}")

## 6. Model Verification

In [ ]:
# Verify the trained model
model_path = trainer.save_model_path
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location='cpu')
    print(f"Model loaded from: {model_path}")
else:
    print(f"Model not found at: {model_path}")

In [ ]:
# Test prediction
test_query = {"query": "What is the capital of France?"}
result = router.route_single(test_query)

print(f"Test query: {test_query['query']}")
print(f"Routed to: {result['model_name']}")

## Summary

In this notebook, we:

1. **Loaded Configuration**: Set up GraphRouter with YAML configuration
2. **Understood Graph Structure**: Query-LLM bipartite graph
3. **Trained GNN Model**: Used message passing to learn representations
4. **Verified Model**: Tested routing with sample queries

**Key Takeaways**:
- GraphRouter models query-LLM relationships as a graph
- GNN can capture complex interaction patterns
- Edge masking during training improves generalization

**Next Steps**:
- Use the next part of notebook for inference
- Experiment with different GNN architectures

# GraphRouter - Inference

This part of notebook demonstrates how to use a trained **GraphRouter** for inference.

## 1. Environment Setup (optional)

In [ ]:
# Install required packages (for Colab)
!pip install llmrouter-lib torch torch-geometric
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
import torch
from llmrouter.models.graphrouter import GraphRouter
from llmrouter.utils import setup_environment
import yaml

setup_environment()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Load Trained Router

In [ ]:
CONFIG_PATH = "configs/model_config_train/graphrouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

router = GraphRouter(yaml_path=CONFIG_PATH)
print(f"Router loaded with {len(router.llm_data)} LLM candidates")

## 3. Query Routing

In [ ]:
EXAMPLE_QUERIES = [
    {"query": "What is the capital of France?"},
    {"query": "Solve the equation: 2x + 5 = 15"},
    {"query": "Write a Python function to check if a number is prime."},
    {"query": "Explain quantum computing in simple terms."},
]

print("Routing Results:")
print("=" * 60)

for i, query in enumerate(EXAMPLE_QUERIES, 1):
    result = router.route_single(query)
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result['model_name']}")

## 4. File-Based Inference

Load queries from a file and save results.

In [ ]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/graphrouter_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")
    
    # Save results
    save_results_to_file(file_results, OUTPUT_FILE)
    
    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

## Summary

This notebook demonstrated:
1. Loading a trained GraphRouter
2. Routing queries using GNN-based inference

GraphRouter is effective for:
- Capturing relational patterns between queries and LLMs
- Leveraging graph structure for better routing